# Non-Clifford pipeline (CircLS)

This notebook slowly goes over the process of:
1. Creating a small logical circuit with Qiskit.
2. Writing the circuit as the OpenQASM 2 program that CircLS reads.
3. Turning the gate list into a Pauli-based circuit (NWQEC front end): S-proxy rotations and Pauli product measurements.
4. Rewriting that list into CircLS's canonical form (terminal-measurement re-selection).
5. Compiling the rotations and the measurements into surface-code patches, |Y⟩ ancillas, corridors and merge windows (CircLS).
6. Reading the stim circuit: every `DETECTOR` comes from LightStim's syndrome tracker; the `OBSERVABLE_INCLUDE`s are the tracker's deterministic readouts plus one per deterministic program bit.
7. Simulating the circuit: fault distance, logical error rate.

## 1. Creating a Qiskit circuit

The circuit is a [Toffoli gate](https://en.wikipedia.org/wiki/Toffoli_gate) on |110⟩, decomposed by Qiskit into H, T, T† and CNOT.

In [1]:
from qiskit import QuantumCircuit, transpile

def toffoli_encoding(circuit_name: str, draw_circuit: bool = False) -> QuantumCircuit:
    """Create the Toffoli circuit on |110> in the H, T, T†, CNOT gate set."""
    qc = QuantumCircuit(3, name=circuit_name)
    for q in range(3):
        qc.reset(q)
    qc.x(0)
    qc.x(1)
    qc.ccx(0, 1, 2)
    qc = transpile(qc, basis_gates=["h", "t", "tdg", "cx", "x", "s", "sdg"], optimization_level=0)
    if draw_circuit:
        print(f"\n======> QISKIT circuit: {circuit_name.upper()}.\n", qc)
    return qc

circuit_name = "toffoli"
qc = toffoli_encoding(circuit_name, draw_circuit=True)


======> QISKIT circuit: TOFFOLI.
           ┌───┐                                             ┌───┐      
q_0: ─|0>─┤ X ├──────────────■─────────────────────■────■───┤ T ├───■──
          ├───┤              │             ┌───┐   │  ┌─┴─┐┌┴───┴┐┌─┴─┐
q_1: ─|0>─┤ X ├──■───────────┼─────────■───┤ T ├───┼──┤ X ├┤ Tdg ├┤ X ├
          ├───┤┌─┴─┐┌─────┐┌─┴─┐┌───┐┌─┴─┐┌┴───┴┐┌─┴─┐├───┤└┬───┬┘└───┘
q_2: ─|0>─┤ H ├┤ X ├┤ Tdg ├┤ X ├┤ T ├┤ X ├┤ Tdg ├┤ X ├┤ T ├─┤ H ├──────
          └───┘└───┘└─────┘└───┘└───┘└───┘└─────┘└───┘└───┘ └───┘      


## 2. The OpenQASM 2 program that CircLS reads

In [2]:
from qiskit import qasm2, ClassicalRegister

# CIRCUIT -> CIRCLS PROGRAM: the Section 1 circuit, read out in Z
qc_circls = qc.copy()
qc_circls.add_register(ClassicalRegister(3, "c"))
qc_circls.measure([0, 1, 2], [0, 1, 2])

# QISKIT -> QASM
toffoli_qasm_str = qasm2.dumps(qc_circls)
print(toffoli_qasm_str)
qasm = toffoli_qasm_str

OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
creg c[3];
reset q[0];
x q[0];
reset q[1];
x q[1];
reset q[2];
h q[2];
cx q[1],q[2];
tdg q[2];
cx q[0],q[2];
t q[2];
cx q[1],q[2];
t q[1];
tdg q[2];
cx q[0],q[2];
cx q[0],q[1];
t q[0];
tdg q[1];
cx q[0],q[1];
t q[2];
h q[2];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];


## 3. NWQEC: from gates to Pauli rotations and Pauli product measurements

CircLS's pipeline is Clifford. With `t_as_s` every T gate is replaced by an S gate, whose resource state is a |Y⟩ ancilla.

In [3]:
import sys, os, pathlib, contextlib, io

# the CircLS checkout: the first directory at or above the working directory that holds both packages
ROOT = next(d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "circls").is_dir() and (d / "lightstim").is_dir())
for p in (ROOT, ROOT / "experiments", ROOT / "gallery" / "tools"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from circls.interop.nwqec.frontend import load_clifford_t_as_s
from nb_helpers import fmt_ops_plain

pbc = load_clifford_t_as_s(qasm)           # the Pauli-based circuit: rotations, then terminal measurements
print("\n".join(fmt_ops_plain(pbc)))

(+IZX)_pi/4 rotation
(+ZZX)_pi/4 rotation
(-IZI)_pi/4 rotation
(+ZIX)_pi/4 rotation
(-ZII)_pi/4 rotation
(-ZZI)_pi/4 rotation
(+IIX)_pi/4 rotation
(-ZII) measurement
(-IZI) measurement
(+IIZ) measurement


## 4. Canonical form: terminal-measurement re-selection

In [4]:
from circls.compiler.measure_reduce import reduce_measurements
from circls.compiler.scheduling import schedule_ops
from circls.interop.ir.pauli_ir import PauliCircuit
from circls.pipeline import reorder_weight1_last, drop_free_prefix, y_free_sweep, expand_gadgets

rotations = [o for o in pbc.ops if o.kind != "m"]
m_only = PauliCircuit(pbc.num_qubits, [o for o in pbc.ops if o.kind == "m"])
reduced, reconstruction = reduce_measurements(m_only)
ordered, _ = reorder_weight1_last(reduced)
scheduled, _ = schedule_ops(drop_free_prefix(y_free_sweep(PauliCircuit(pbc.num_qubits, rotations + list(ordered.ops)))), parallel=True)
program = expand_gadgets(scheduled)
names = {q: (f"q{q}" if q < pbc.num_qubits else f"y{q - pbc.num_qubits}") for q in range(program.num_qubits)}
for op in program.ops:
    print(f"{op.kind:7s} " + "  ".join(f"{names[q]}.{p}" for q, p in op.targets.items()))

init_y  y0.Y
mpp     q0.Z  y0.Z
mpp     y0.X
init_y  y1.Y
mpp     q1.Z  q2.X  y1.Z
mpp     y1.X
init_y  y2.Y
mpp     q0.Z  q1.Z  q2.X  y2.Z
mpp     y2.X
init_y  y3.Y
mpp     q1.Z  y3.Z
mpp     y3.X
init_y  y4.Y
mpp     q0.Z  q2.X  y4.Z
mpp     y4.X
init_y  y5.Y
mpp     q0.Z  q1.Z  y5.Z
mpp     y5.X
init_y  y6.Y
mpp     q2.X  y6.Z
mpp     y6.X
mpp     q0.Z
mpp     q1.Z
mpp     q2.Z


## 5. CircLS compilation: patches, corridors, merge windows

Compiled with re-selection (the default) and `t_as_s=True`.

In [5]:
import circls.pipeline as P
from circls.metrics.experiment_stats import experiment_stats
from nb_helpers import ppm_step_lines

def compile_circuit(distance, reduce=True):
    with contextlib.redirect_stdout(io.StringIO()):
        return P.compile_qasm(qasm, distance=distance, measure_reduction=reduce, t_as_s=True)

cp3 = compile_circuit(3)
exp = cp3.experiment
SEP = "-" * 80
print("placement (patch -> tile):", cp3.placement)
print(SEP)
print("orientation             :", {s.name: s.orientation for s in exp.patches})
print(SEP)
print("initial / final bases   :", dict(sorted(exp.initial_states.items())), "/", dict(sorted(exp.final_measure_states.items())))
print(SEP)
for line, route in zip(ppm_step_lines(exp.ppm_sequence), exp._routes):
    print("merge window            :", line, "  corridor tiles:", sorted(route.tree))
print(SEP)
print(f"spacetime volume        : {experiment_stats(exp, cp3.circuit).volume_blocks:.1f} blocks of d x d x d")

placement (patch -> tile): {'q0': (1, 1), 'q1': (3, 1), 'q2': (5, 1), 'y0': (7, 1), 'y1': (1, 3), 'y2': (3, 3), 'y3': (5, 3), 'y4': (7, 3), 'y5': (1, 5), 'y6': (3, 5)}
--------------------------------------------------------------------------------
orientation             : {'q0': 'X_horizontal', 'q1': 'X_horizontal', 'q2': 'X_vertical', 'y0': 'X_vertical', 'y1': 'X_vertical', 'y2': 'X_vertical', 'y3': 'X_vertical', 'y4': 'X_vertical', 'y5': 'X_vertical', 'y6': 'X_vertical'}
--------------------------------------------------------------------------------
initial / final bases   : {'q0': 'Z', 'q1': 'Z', 'q2': 'Z', 'y0': 'Y', 'y1': 'Y', 'y2': 'Y', 'y3': 'Y', 'y4': 'Y', 'y5': 'Y', 'y6': 'Y'} / {'q0': 'Z', 'q1': 'Z', 'q2': 'Z', 'y0': 'X', 'y1': 'X', 'y2': 'X', 'y3': 'X', 'y4': 'X', 'y5': 'X', 'y6': 'X'}
--------------------------------------------------------------------------------
merge window            : PPM 0: q0.Z  y0.Z   corridor tiles: [(2, 0), (2, 1), (3, 0), (4, 0), (5, 0), (6, 0

## 6. The stim circuit and where its annotations come from

No `DETECTOR` or `OBSERVABLE_INCLUDE` is written by hand: the detectors come from LightStim's syndrome tracker, the observables are the tracker's deterministic readouts plus one per deterministic program bit.

In [6]:
circuit = cp3.circuit
print(f"d = 3: {circuit.num_qubits} qubits, {circuit.num_ticks} ticks, {circuit.num_measurements} measurements, "
      f"{circuit.num_detectors} detectors, {circuit.num_observables} observables")

d = 3: 742 qubits, 372 ticks, 4105 measurements, 3365 detectors, 5 observables


Merge window `PPM 1`, tick by tick.

In [7]:
from IPython.display import SVG

# merge window PPM 1: corridor reset at tick 93, corridor readout at tick 125
display(SVG(str(circuit.diagram("detslice-with-ops-svg", tick=range(93, 126)))))

## 7. Simulation: fault distance and logical error rate

Uniform circuit-level depolarising noise: graphlike fault distance and logical error rate (PyMatching, 200 000 shots).

In [8]:
from noise_inject import inject_uniform_noise

for d in (3, 5):
    cp = compile_circuit(d)
    noisy = inject_uniform_noise(cp.circuit, 1e-3)
    dist = len(noisy.shortest_graphlike_error(ignore_ungraphlike_errors=False))
    print(f"d = {d}: {cp.circuit.num_qubits:4d} qubits, {cp.circuit.num_detectors:4d} detectors, "
          f"{cp.circuit.num_observables} observables, graphlike fault distance = {dist}")

d = 3:  742 qubits, 3365 detectors, 5 observables, graphlike fault distance = 3


d = 5: 1868 qubits, 14506 detectors, 5 observables, graphlike fault distance = 5


In [9]:
import pymatching
import numpy as np

SHOTS = 200_000
for d in (3, 5):
    cp = compile_circuit(d)
    for p in (5e-4, 1e-3):
        noisy = inject_uniform_noise(cp.circuit, p)
        matcher = pymatching.Matching.from_detector_error_model(noisy.detector_error_model(decompose_errors=True))
        det, obs = noisy.compile_detector_sampler(seed=0).sample(SHOTS, separate_observables=True)
        errors = int(np.any(matcher.decode_batch(det) != obs, axis=1).sum())
        print(f"d = {d}, p = {p:.0e}: LER = {errors / SHOTS:.2e}  ({errors} / {SHOTS})")

d = 3, p = 5e-04: LER = 1.39e-01  (27858 / 200000)


d = 3, p = 1e-03: LER = 3.82e-01  (76467 / 200000)


d = 5, p = 5e-04: LER = 2.08e-02  (4163 / 200000)


d = 5, p = 1e-03: LER = 1.39e-01  (27723 / 200000)
